In [178]:
import pandas as pd 
import numpy as np


import yfinance as y
import seaborn as sns 
import matplotlib.pyplot as plt

import warnings 
warnings.filterwarnings("ignore")

sns.set()


from pyspark.sql.window import Window
from pyspark.sql.functions import *


In [190]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ProjetoLocal") \
    .master("local[*]") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

# spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 100)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 0)

# silenciando os avisos do spark
spark.sparkContext.setLogLevel("ERROR")



In [191]:
df = spark.read.parquet('../data/bronze/questionario_socioeconomico_2023.parquet', inferSchema=True)

##### **RENOMEANDO COLUNAS**
Padrão de nomenclatura da origem dos dados é poucointuitivodificultando a interpretação por parte do usuário

In [192]:
mapeamento_colunas = {
    "Q001": "esc_pai",
    "Q002": "esc_mae",
    "Q003": "ocupacao_pai",
    "Q004": "ocupacao_mae",
    "Q005": "num_pessoas_residencia",
    "Q006": "renda_familiar",
    "Q007": "empregado_domestico",
    "Q008": "qtd_banheiros",
    "Q009": "qtd_quartos",
    "Q010": "tem_carro",
    "Q011": "tem_moto",
    "Q012": "tem_maquina_lavar",
    "Q013": "tem_geladeira",
    "Q014": "tem_freezer",
    "Q016": "tem_maquina_lavar_louca",
    "Q018": "tem_aspirador_po",
    "Q019": "tem_tv_cores",
    "Q021": "tem_tv_assinatura_dvd",
    "Q022": "qtd_celulares",
    "Q023": "tem_telefone_fixo",
    "Q024": "tem_computador",
    "Q025": "tem_internet"
}


df_renomeado = df
for coluna_antiga, coluna_nova in mapeamento_colunas.items():
    # Verifica se a coluna realmente existe no DataFrame antes de tentar renomear
    if coluna_antiga in df_renomeado.columns:
        df_renomeado = df_renomeado.withColumnRenamed(coluna_antiga, coluna_nova)

In [193]:
df_renomeado.count(), len(df_renomeado.columns)

(3933955, 22)

In [194]:
df_renomeado.limit(1)

esc_pai,esc_mae,ocupacao_pai,ocupacao_mae,num_pessoas_residencia,renda_familiar,empregado_domestico,qtd_banheiros,qtd_quartos,tem_carro,tem_moto,tem_maquina_lavar,tem_geladeira,tem_freezer,tem_maquina_lavar_louca,tem_aspirador_po,tem_tv_cores,tem_tv_assinatura_dvd,qtd_celulares,tem_telefone_fixo,tem_computador,tem_internet
"Completou o Ensino Médio, mas não completou a Faculdade.","Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio.","Diarista, empregado doméstico, cuidador de idosos, babá, cozinheiro (em casas particulares), motorista particular, jardineiro, faxineiro de empresas e prédios, vigilante, porteiro, carteiro, office-boy, vendedor, caixa, atendente de loja, auxiliar administrativo, recepcionista, servente de pedreiro, repositor de mercadoria.","Diarista, empregada doméstica, cuidadora de idosos, babá, cozinheira (em casas particulares), motorista particular, jardineira, faxineira de empresas e prédios, vigilante, porteira, carteira, office-boy, vendedora, caixa, atendente de loja, auxiliar administrativa, recepcionista, servente de pedreiro, repositora de mercadoria.",4,"Até R$ 1.320,00",Não.,"Sim, um.","Sim, dois.",Não.,Não.,"Sim, uma.",Não.,Não.,"Sim, um.",Não.,"Sim, uma.",Não.,"Sim, um.",Não.,Não.,Sim.


In [202]:
(
    df_renomeado

        .select(
            col('empregado_domestico')
        )
        .distinct()
)

empregado_domestico
Não.
"Sim, três ou quatro dias por semana."
"Sim, pelo menos cinco dias por semana."
"Sim, um ou dois dias por semana."


##### **COLUNA ESCOLALIDADE**

In [ ]:
mapping_escolaridade = {
    "Nunca estudou.": "1. Nunca estudou",
    "Não completou a 4ª série/5º ano do Ensino Fundamental.": "2. Fund. I Incompleto",
    "Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental.": "3. Fund. I Comp. / Fund. II Incomp.",
    "Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio.": "4. Fund. II Comp. / Médio Incomp.",
    "Completou o Ensino Médio, mas não completou a Faculdade.": "5. Ensino Médio Completo",
    "Completou a Faculdade, mas não completou a Pós-graduação.": "6. Superior Completo",
    "Completou a Pós-graduação.": "7. Pós-graduação",
    "Não sei.": "8. Não sabe / Ignorado"
}



df_teste = (

    # TRANDO COLUNA DE ESCOLARLIDADE TANTO DO PAI QUANTO DA MAE
    df_renomeado
        .replace(
            mapping_escolaridade, subset=['esc_pai', 'esc_mae']
        )


        # TRANTANDO COLUNA PESSOAS NA RESIDENCIA
        .withColumn('num_pessoas_residencia',
                            when(col('num_pessoas_residencia').contains(','), 
                                 split(col('num_pessoas_residencia'),',' )[0] )
                                 .otherwise(col('num_pessoas_residencia') )
                            )
)

In [201]:
(
    df_teste    
        .select(
            col('num_pessoas_residencia')
        )
        .distinct()
)

num_pessoas_residencia
7
15
11
3
8
16
5
18
17
6
